In [ ]:
# Algorithm 1
import numpy as np
import heapq
from math import log2


# ============================================================
# NONLINEAR REVERSIBLE REPRESENTATION BENCHMARK
#
# K = 12
#
# Five heterogeneous low-entropy latent coordinates:
#
#   p = [.001, .005, .01, .02, .05]
#
# Seven uniform coordinates.
#
# The hidden nonlinear representation is:
#
#   A_i = Y_i XOR (Y5 AND Y6) XOR Y(7+i)
#
# for i = 0,...,4
#
# and:
#
#   A5...A11 = Y5...Y11
#
# This is generated by reversible gates:
#
#   TOF(i <- 5 & 6), i=0..4
#   CNOT(i <- 7+i), i=0..4
#
# Every output coordinate is close to 1 bit.
#
# Every low-entropy direction is NONLINEARLY hidden.
#
# The benchmark compares:
#
#   1. pairwise MI
#   2. exact best linear GF(2) transform
#   3. nonlinear reversible beam search
#
# and verifies:
#
#   - exact reversibility
#   - joint entropy invariance
#   - held-out generalization
#   - distance from planted optimum
#
# ============================================================


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 20260907

K = 12

TRUE_LOW_RANK = 5

P_LOW = np.array(
    [
        0.001,
        0.005,
        0.010,
        0.020,
        0.050,
    ],
    dtype=float
)

assert len(P_LOW) == TRUE_LOW_RANK

N_TRAIN = 8192
N_VALIDATION = 16384
N_TEST = 65536

BEAM_WIDTH = 64
MAX_DEPTH = 10


# ============================================================
# INFORMATION THEORY
# ============================================================

def entropy_binary(x):
    x = np.asarray(
        x,
        dtype=np.uint8
    )

    p = float(
        np.mean(x)
    )

    if p <= 0.0 or p >= 1.0:
        return 0.0

    return -(
        p * log2(p)
        + (1.0 - p)
        * log2(1.0 - p)
    )


def binary_entropy(p):
    p = float(p)

    if p <= 0.0 or p >= 1.0:
        return 0.0

    return -(
        p * log2(p)
        + (1.0 - p)
        * log2(1.0 - p)
    )


# ============================================================
# GF(2) MATRIX ALGEBRA
# ============================================================

def gf2_matmul(A, B):
    return (
        (
            A.astype(np.int32)
            @ B.astype(np.int32)
        ) & 1
    ).astype(np.uint8)


def gf2_rank(A):
    A = np.asarray(
        A,
        dtype=np.uint8
    ).copy()

    rows, cols = A.shape

    pivot_row = 0

    for col in range(cols):

        pivots = np.flatnonzero(
            A[pivot_row:, col]
        )

        if len(pivots) == 0:
            continue

        pivot = int(
            pivot_row + pivots[0]
        )

        if pivot != pivot_row:

            A[
                [pivot_row, pivot]
            ] = A[
                [pivot, pivot_row]
            ]

        for r in range(rows):

            if (
                r != pivot_row
                and A[r, col]
            ):
                A[r] ^= A[pivot_row]

        pivot_row += 1

        if pivot_row == rows:
            break

    return pivot_row


def gf2_inverse(A):
    A = np.asarray(
        A,
        dtype=np.uint8
    )

    n, m = A.shape

    assert n == m

    aug = np.concatenate(
        [
            A.copy(),
            np.eye(
                n,
                dtype=np.uint8
            )
        ],
        axis=1
    )

    for col in range(n):

        pivots = np.flatnonzero(
            aug[col:, col]
        )

        if len(pivots) == 0:
            raise ValueError(
                "Singular GF(2) matrix."
            )

        pivot = int(
            col + pivots[0]
        )

        if pivot != col:

            aug[
                [col, pivot]
            ] = aug[
                [pivot, col]
            ]

        for r in range(n):

            if (
                r != col
                and aug[r, col]
            ):
                aug[r] ^= aug[col]

    inv = aug[:, n:]

    assert np.array_equal(
        gf2_matmul(
            A,
            inv
        ),
        np.eye(
            n,
            dtype=np.uint8
        )
    )

    return inv


# ============================================================
# INTEGER GF(2) BASIS
# ============================================================

def integer_basis(vectors):
    pivots = {}

    basis = []

    for raw in vectors:

        x = int(raw)

        while x:

            pivot = (
                x.bit_length() - 1
            )

            if pivot in pivots:

                x ^= pivots[pivot]

            else:

                pivots[pivot] = x
                basis.append(x)

                break

    return basis


def vector_to_mask(v):
    value = 0

    for j, bit in enumerate(v):

        if int(bit):
            value |= (
                1 << j
            )

    return value


def mask_to_vector(
    mask,
    K
):
    return np.array(
        [
            (int(mask) >> j) & 1
            for j in range(K)
        ],
        dtype=np.uint8
    )


def format_relation(
    mask,
    K
):
    terms = []

    for j in range(K):

        if (
            (int(mask) >> j)
            & 1
        ):
            terms.append(
                f"A{j}"
            )

    return (
        " XOR ".join(terms)
        if terms
        else "0"
    )


# ============================================================
# DATA GENERATION
# ============================================================

def generate_latent(
    K,
    N,
    p_low,
    rng
):
    Y = np.zeros(
        (
            K,
            N
        ),
        dtype=np.uint8
    )

    # Low entropy channels.
    for i, p in enumerate(
        p_low
    ):

        Y[i] = (
            rng.random(N)
            < p
        ).astype(
            np.uint8
        )

    # Uniform channels.
    for i in range(
        len(p_low),
        K
    ):

        Y[i] = rng.integers(
            0,
            2,
            size=N,
            dtype=np.uint8
        )

    return Y


# ============================================================
# REVERSIBLE BOOLEAN GATES
# ============================================================

def apply_gate(
    B,
    gate
):
    B = B.copy()

    kind = gate[0]

    if kind == "NOT":

        t = gate[1]

        B[t] ^= 1

    elif kind == "CNOT":

        t = gate[1]
        s = gate[2]

        B[t] ^= B[s]

    elif kind == "TOF":

        t = gate[1]
        c1 = gate[2]
        c2 = gate[3]

        B[t] ^= (
            B[c1]
            & B[c2]
        )

    else:

        raise ValueError(
            f"Unknown gate: {gate}"
        )

    return B


def apply_gate_sequence(
    B,
    gates
):
    result = B

    for gate in gates:

        result = apply_gate(
            result,
            gate
        )

    return result


# ============================================================
# PLANTED NONLINEAR CIRCUIT
# ============================================================

def planted_forward_circuit():

    gates = []

    # First hide each low entropy bit with a nonlinear
    # Toffoli term.
    #
    # All controls are uniform Y5,Y6.

    for i in range(
        TRUE_LOW_RANK
    ):

        gates.append(
            (
                "TOF",
                i,
                5,
                6
            )
        )

    # Then mask each target with a distinct uniform bit.
    #
    # These masks make every target marginal close to uniform.

    for i in range(
        TRUE_LOW_RANK
    ):

        gates.append(
            (
                "CNOT",
                i,
                7 + i
            )
        )

    return tuple(gates)


# ============================================================
# PAIRWISE MI
# ============================================================

def mutual_information_binary(
    x,
    y
):
    counts = np.zeros(
        (2, 2),
        dtype=np.int64
    )

    for a, b in zip(
        x,
        y
    ):

        counts[
            int(a),
            int(b)
        ] += 1

    N = len(x)

    pxy = counts / N

    px = pxy.sum(
        axis=1
    )

    py = pxy.sum(
        axis=0
    )

    mi = 0.0

    for i in range(2):

        for j in range(2):

            if pxy[i, j] > 0:

                mi += (
                    pxy[i, j]
                    * log2(
                        pxy[i, j]
                        / (
                            px[i]
                            * py[j]
                        )
                    )
                )

    return float(mi)


def pairwise_mi_summary(A):

    K = A.shape[0]

    values = []

    for i in range(K):

        for j in range(
            i + 1,
            K
        ):

            values.append(
                mutual_information_binary(
                    A[i],
                    A[j]
                )
            )

    values = np.asarray(
        values
    )

    return {
        "mean": float(
            values.mean()
        ),
        "median": float(
            np.median(values)
        ),
        "max": float(
            values.max()
        ),
    }


# ============================================================
# JOINT ENTROPY
# ============================================================

def joint_entropy_binary(A):

    K, N = A.shape

    if K > 16:

        raise ValueError(
            "Exact joint entropy routine is intended for K <= 16."
        )

    codes = np.zeros(
        N,
        dtype=np.uint32
    )

    for j in range(K):

        codes |= (
            A[j].astype(
                np.uint32
            )
            << j
        )

    counts = np.bincount(
        codes,
        minlength=1 << K
    )

    counts = counts[
        counts > 0
    ]

    p = (
        counts
        / N
    )

    return float(
        -np.sum(
            p
            * np.log2(p)
        )
    )


# ============================================================
# LINEAR RELATION ENTROPY
# ============================================================

def relation_entropy(
    A,
    mask
):
    result = np.zeros(
        A.shape[1],
        dtype=np.uint8
    )

    bits = int(mask)

    while bits:

        lsb = (
            bits
            & -bits
        )

        j = (
            lsb.bit_length()
            - 1
        )

        result ^= A[j]

        bits ^= lsb

    return entropy_binary(
        result
    )


# ============================================================
# EXACT BEST LINEAR GF(2) TRANSFORM
# ============================================================

def exact_best_linear_transform(
    A_train,
    A_test
):
    K = A_train.shape[0]

    masks = list(
        range(
            1,
            1 << K
        )
    )

    costs = {}

    for mask in masks:

        costs[mask] = (
            relation_entropy(
                A_train,
                mask
            )
        )

    ordered = sorted(
        masks,
        key=lambda m:
            costs[m]
    )

    basis = []

    for mask in ordered:

        old_rank = len(
            integer_basis(
                basis
            )
        )

        new_rank = len(
            integer_basis(
                basis + [mask]
            )
        )

        if new_rank > old_rank:

            basis.append(
                mask
            )

            if len(basis) == K:
                break

    T = np.stack(
        [
            mask_to_vector(
                m,
                K
            )
            for m in basis
        ]
    )

    assert gf2_rank(T) == K

    B_train = gf2_matmul(
        T,
        A_train
    )

    B_test = gf2_matmul(
        T,
        A_test
    )

    h_train = np.array(
        [
            entropy_binary(
                B_train[i]
            )
            for i in range(K)
        ]
    )

    h_test = np.array(
        [
            entropy_binary(
                B_test[i]
            )
            for i in range(K)
        ]
    )

    return {
        "T": T,
        "basis": basis,
        "train_B": B_train,
        "test_B": B_test,
        "train_entropies":
            h_train,
        "test_entropies":
            h_test,
        "train_sum":
            float(h_train.sum()),
        "test_sum":
            float(h_test.sum()),
    }


# ============================================================
# PACKED BOOLEAN REPRESENTATION
# ============================================================

POP8 = np.array(
    [
        bin(i).count("1")
        for i in range(256)
    ],
    dtype=np.uint8
)


def pack_binary(A):

    return np.packbits(
        A,
        axis=1,
        bitorder="little"
    )


def packed_entropy(
    row,
    N
):
    ones = int(
        POP8[row].sum()
    )

    p = (
        ones / N
    )

    if p <= 0.0 or p >= 1.0:
        return 0.0

    return -(
        p * log2(p)
        + (1.0 - p)
        * log2(1.0 - p)
    )


# ============================================================
# PACKED GATE
# ============================================================

def apply_packed_gate(
    P,
    gate
):
    Q = P.copy()

    kind = gate[0]

    if kind == "NOT":

        t = gate[1]

        Q[t] ^= 0xFF

    elif kind == "CNOT":

        t = gate[1]
        s = gate[2]

        Q[t] ^= Q[s]

    elif kind == "TOF":

        t = gate[1]
        c1 = gate[2]
        c2 = gate[3]

        Q[t] ^= (
            Q[c1]
            & Q[c2]
        )

    else:

        raise ValueError(
            f"Unknown gate: {gate}"
        )

    return Q


# ============================================================
# GATE LIBRARY
# ============================================================

def generate_gate_library(K):

    gates = []

    # NOT
    for t in range(K):

        gates.append(
            ("NOT", t)
        )

    # CNOT
    for t in range(K):

        for s in range(K):

            if t == s:
                continue

            gates.append(
                (
                    "CNOT",
                    t,
                    s
                )
            )

    # TOFFOLI
    for t in range(K):

        controls = [
            c
            for c in range(K)
            if c != t
        ]

        for a in range(
            len(controls)
        ):

            for b in range(
                a + 1,
                len(controls)
            ):

                gates.append(
                    (
                        "TOF",
                        t,
                        controls[a],
                        controls[b]
                    )
                )

    return gates


# ============================================================
# BEAM STATE
# ============================================================

class BeamState:

    __slots__ = (
        "P",
        "entropies",
        "score",
        "path",
    )

    def __init__(
        self,
        P,
        entropies,
        score,
        path
    ):
        self.P = P
        self.entropies = entropies
        self.score = float(score)
        self.path = tuple(path)


# ============================================================
# NONLINEAR BEAM SEARCH
# ============================================================

def nonlinear_beam_search(
    A_train,
    K,
    beam_width,
    max_depth
):
    N = A_train.shape[1]

    gates = generate_gate_library(
        K
    )

    P = pack_binary(
        A_train
    )

    initial_entropies = np.array(
        [
            packed_entropy(
                P[i],
                N
            )
            for i in range(K)
        ]
    )

    initial = BeamState(
        P=P,
        entropies=initial_entropies,
        score=float(
            initial_entropies.sum()
        ),
        path=()
    )

    beam = [
        initial
    ]

    best_by_depth = {
        0: initial
    }

    counter = 0

    print()
    print("=" * 115)
    print(
        "NONLINEAR REVERSIBLE BEAM SEARCH"
    )
    print("=" * 115)

    print()
    print(
        f"gate library = "
        f"{len(gates)} gates"
    )

    print(
        f"beam width = "
        f"{beam_width}"
    )

    print(
        f"max depth = "
        f"{max_depth}"
    )

    for depth in range(
        1,
        max_depth + 1
    ):

        # Root of this max-heap stores the worst retained
        # candidate.
        heap = []

        for state in beam:

            last_gate = (
                state.path[-1]
                if state.path
                else None
            )

            for gate in gates:

                # Every gate is self-inverse, so immediately
                # repeating the same gate just returns to the
                # parent state.
                if (
                    last_gate is not None
                    and gate == last_gate
                ):
                    continue

                target = gate[1]

                candidate_P = (
                    apply_packed_gate(
                        state.P,
                        gate
                    )
                )

                target_entropy = (
                    packed_entropy(
                        candidate_P[target],
                        N
                    )
                )

                new_score = (
                    state.score
                    - state.entropies[target]
                    + target_entropy
                )

                candidate_entropies = (
                    state.entropies.copy()
                )

                candidate_entropies[
                    target
                ] = target_entropy

                candidate = BeamState(
                    P=candidate_P,
                    entropies=candidate_entropies,
                    score=new_score,
                    path=(
                        state.path
                        + (gate,)
                    )
                )

                counter += 1

                # Negative score makes the largest actual
                # entropy the minimum heap key, so it becomes
                # the first item popped when the heap is too big.
                item = (
                    -new_score,
                    counter,
                    candidate
                )

                heapq.heappush(
                    heap,
                    item
                )

                if len(heap) > beam_width:

                    heapq.heappop(
                        heap
                    )

        if not heap:
            break

        beam = [
            item[2]
            for item in heap
        ]

        beam.sort(
            key=lambda s:
                s.score
        )

        best = beam[0]

        best_by_depth[
            depth
        ] = best

        print(
            f"depth {depth:2d}: "
            f"train entropy = "
            f"{best.score:.8f}"
        )

        print(
            "           path =",
            best.path
        )

    return (
        best_by_depth,
        gates
    )


# ============================================================
# VALIDATION-BASED DEPTH SELECTION
# ============================================================

def select_depth(
    best_by_depth,
    A_validation,
    K,
    gate_count
):
    candidates = []

    for depth, state in sorted(
        best_by_depth.items()
    ):

        B = apply_gate_sequence(
            A_validation,
            state.path
        )

        entropy = float(
            sum(
                entropy_binary(
                    B[i]
                )
                for i in range(K)
            )
        )

        complexity = (
            depth
            * log2(gate_count)
            / A_validation.shape[1]
        )

        score = (
            entropy
            + complexity
        )

        candidates.append(
            {
                "depth": depth,
                "path": state.path,
                "validation_entropy":
                    entropy,
                "complexity":
                    complexity,
                "score":
                    score,
            }
        )

    selected = min(
        candidates,
        key=lambda x:
            x["score"]
    )

    print()
    print("=" * 110)
    print(
        "NONLINEAR DEPTH SELECTION"
    )
    print("=" * 110)

    print()

    for c in candidates:

        print(
            f"depth {c['depth']:2d}: "
            f"validation="
            f"{c['validation_entropy']:.8f} "
            f"complexity="
            f"{c['complexity']:.8f} "
            f"score="
            f"{c['score']:.8f}"
        )

    print()
    print(
        f"SELECTED DEPTH = "
        f"{selected['depth']}"
    )

    return selected


# ============================================================
# GATE DISPLAY
# ============================================================

def format_gate(gate):

    if gate[0] == "NOT":

        return (
            f"NOT({gate[1]})"
        )

    if gate[0] == "CNOT":

        return (
            f"CNOT({gate[1]} <- {gate[2]})"
        )

    return (
        f"TOF({gate[1]} <- "
        f"{gate[2]} & {gate[3]})"
    )


def print_path(
    name,
    path
):
    print()
    print(name)

    for i, gate in enumerate(
        path
    ):

        print(
            f"  {i+1:02d}. "
            f"{format_gate(gate)}"
        )

def verify_path_reversal(path, data):
    """
    Verifies that applying the path's gates and then applying 
    them in reverse perfectly reconstructs the data.
    """
    # 1. Apply the sequence of gates
    forward_result = apply_gate_sequence(data, path)
    
    # 2. Reverse the sequence of gates (inverse circuit)
    inverse_path = tuple(reversed(path))
    
    # 3. Apply the reversed sequence
    reconstructed = apply_gate_sequence(forward_result, inverse_path)
    
    # 4. Check if it exactly matches the original input
    return bool(np.array_equal(data, reconstructed))

# ============================================================
# MAIN
# ============================================================

def main():

    rng = np.random.default_rng(
        SEED
    )

    print()
    print("=" * 120)
    print(
        "CORRECTED NONLINEAR HIGHER-ORDER DEPENDENCE BENCHMARK"
    )
    print("=" * 120)

    print()
    print(
        f"K = {K}"
    )

    print(
        f"TRUE LOW-ENTROPY RANK = "
        f"{TRUE_LOW_RANK}"
    )

    print(
        f"P_LOW = "
        f"{P_LOW}"
    )

    # --------------------------------------------------------
    # Theoretical latent entropy.
    # --------------------------------------------------------

    residual_entropies = np.array(
        [
            binary_entropy(p)
            for p in P_LOW
        ]
    )

    theoretical_optimum = (
        K
        - TRUE_LOW_RANK
        + residual_entropies.sum()
    )

    print()
    print(
        "THEORETICAL LOW-ENTROPY CHANNELS"
    )

    for i, (p, h) in enumerate(
        zip(
            P_LOW,
            residual_entropies
        )
    ):

        print(
            f"  Y{i}: "
            f"p={p:.3f} "
            f"H={h:.8f}"
        )

    print()
    print(
        f"THEORETICAL JOINT ENTROPY = "
        f"{theoretical_optimum:.8f}"
    )

    print(
        f"THEORETICAL REDUCTION = "
        f"{100.0 * (
            1.0
            - theoretical_optimum / K
        ):.6f}%"
    )

    # --------------------------------------------------------
    # Planted circuit.
    # --------------------------------------------------------

    planted_path = (
        planted_forward_circuit()
    )

    print_path(
        "PLANTED FORWARD CIRCUIT",
        planted_path
    )

    planted_inverse = tuple(
        reversed(
            planted_path
        )
    )

    print_path(
        "PLANTED INVERSE CIRCUIT",
        planted_inverse
    )

    # --------------------------------------------------------
    # Latent data.
    # --------------------------------------------------------

    Y_train = generate_latent(
        K,
        N_TRAIN,
        P_LOW,
        rng
    )

    Y_validation = generate_latent(
        K,
        N_VALIDATION,
        P_LOW,
        rng
    )

    Y_test = generate_latent(
        K,
        N_TEST,
        P_LOW,
        rng
    )

    # --------------------------------------------------------
    # Observations.
    # --------------------------------------------------------

    A_train = apply_gate_sequence(
        Y_train,
        planted_path
    )

    A_validation = apply_gate_sequence(
        Y_validation,
        planted_path
    )

    A_test = apply_gate_sequence(
        Y_test,
        planted_path
    )

    # --------------------------------------------------------
    # Generator verification.
    # --------------------------------------------------------

    print()
    print("=" * 100)
    print(
        "PLANTED CIRCUIT REVERSIBILITY"
    )
    print("=" * 100)

    print(
        "train =",
        verify_path_reversal(
            planted_path,
            A_train
        )
    )

    print(
        "test  =",
        verify_path_reversal(
            planted_path,
            A_test
        )
    )

    # --------------------------------------------------------
    # Latent empirical entropy.
    #
    # This is an important finite-sample oracle.
    # --------------------------------------------------------

    latent_train_entropies = np.array(
        [
            entropy_binary(
                Y_train[i]
            )
            for i in range(K)
        ]
    )

    latent_test_entropies = np.array(
        [
            entropy_binary(
                Y_test[i]
            )
            for i in range(K)
        ]
    )

    latent_train_sum = float(
        latent_train_entropies.sum()
    )

    latent_test_sum = float(
        latent_test_entropies.sum()
    )

    print()
    print(
        "FINITE-SAMPLE LATENT ORACLE"
    )

    print(
        f"  train entropy = "
        f"{latent_train_sum:.8f}"
    )

    print(
        f"  test entropy = "
        f"{latent_test_sum:.8f}"
    )

    print(
        f"  population optimum = "
        f"{theoretical_optimum:.8f}"
    )

    # --------------------------------------------------------
    # Observed entropy.
    # --------------------------------------------------------

    observed_train_entropies = np.array(
        [
            entropy_binary(
                A_train[i]
            )
            for i in range(K)
        ]
    )

    observed_validation_entropies = np.array(
        [
            entropy_binary(
                A_validation[i]
            )
            for i in range(K)
        ]
    )

    observed_test_entropies = np.array(
        [
            entropy_binary(
                A_test[i]
            )
            for i in range(K)
        ]
    )

    observed_train_sum = float(
        observed_train_entropies.sum()
    )

    observed_validation_sum = float(
        observed_validation_entropies.sum()
    )

    observed_test_sum = float(
        observed_test_entropies.sum()
    )

    print()
    print(
        "OBSERVED REPRESENTATION"
    )

    print(
        f"  train = "
        f"{observed_train_sum:.8f}"
    )

    print(
        f"  validation = "
        f"{observed_validation_sum:.8f}"
    )

    print(
        f"  test = "
        f"{observed_test_sum:.8f}"
    )

    # --------------------------------------------------------
    # Joint entropy.
    # --------------------------------------------------------

    joint_test = joint_entropy_binary(
        A_test
    )

    print()
    print(
        "EMPIRICAL JOINT ENTROPY"
    )

    print(
        f"  observed test = "
        f"{joint_test:.8f}"
    )

    # --------------------------------------------------------
    # Pairwise MI.
    # --------------------------------------------------------

    mi_before = pairwise_mi_summary(
        A_test
    )

    print()
    print(
        "PAIRWISE MI BEFORE"
    )

    print(
        f"  mean = "
        f"{mi_before['mean']:.8f}"
    )

    print(
        f"  median = "
        f"{mi_before['median']:.8f}"
    )

    print(
        f"  max = "
        f"{mi_before['max']:.8f}"
    )

    # ========================================================
    # EXACT LINEAR BASELINE
    # ========================================================

    print()
    print("=" * 110)
    print(
        "EXACT LINEAR GF(2) BASELINE"
    )
    print("=" * 110)

    print()

    linear = exact_best_linear_transform(
        A_train,
        A_test
    )

    print(
        f"linear train entropy = "
        f"{linear['train_sum']:.8f}"
    )

    print(
        f"linear test entropy = "
        f"{linear['test_sum']:.8f}"
    )

    print(
        f"linear test reduction = "
        f"{100.0 * (
            1.0
            - linear['test_sum']
            / observed_test_sum
        ):.6f}%"
    )

    print()
    print(
        "BEST LINEAR BASIS"
    )

    for i, mask in enumerate(
        linear["basis"]
    ):

        print(
            f"  L{i}: "
            f"H={relation_entropy(A_train, mask):.8f} "
            f"{format_relation(mask, K)}"
        )

    # ========================================================
    # NONLINEAR BEAM SEARCH
    # ========================================================

    best_by_depth, gates = (
        nonlinear_beam_search(
            A_train,
            K,
            BEAM_WIDTH,
            MAX_DEPTH
        )
    )

    selected = select_depth(
        best_by_depth,
        A_validation,
        K,
        len(gates)
    )

    learned_path = selected[
        "path"
    ]

    print_path(
        "LEARNED NONLINEAR PATH",
        learned_path
    )

    # --------------------------------------------------------
    # Apply learned transform.
    # --------------------------------------------------------

    B_train = apply_gate_sequence(
        A_train,
        learned_path
    )

    B_validation = apply_gate_sequence(
        A_validation,
        learned_path
    )

    B_test = apply_gate_sequence(
        A_test,
        learned_path
    )

    learned_train_entropies = np.array(
        [
            entropy_binary(
                B_train[i]
            )
            for i in range(K)
        ]
    )

    learned_validation_entropies = np.array(
        [
            entropy_binary(
                B_validation[i]
            )
            for i in range(K)
        ]
    )

    learned_test_entropies = np.array(
        [
            entropy_binary(
                B_test[i]
            )
            for i in range(K)
        ]
    )

    learned_train_sum = float(
        learned_train_entropies.sum()
    )

    learned_validation_sum = float(
        learned_validation_entropies.sum()
    )

    learned_test_sum = float(
        learned_test_entropies.sum()
    )

    # --------------------------------------------------------
    # Reversibility.
    # --------------------------------------------------------

    exact_train = (
        verify_path_reversal(
            learned_path,
            A_train
        )
    )

    exact_test = (
        verify_path_reversal(
            learned_path,
            A_test
        )
    )

    # --------------------------------------------------------
    # Joint entropy after.
    # --------------------------------------------------------

    learned_joint = joint_entropy_binary(
        B_test
    )

    # --------------------------------------------------------
    # Pairwise MI after.
    # --------------------------------------------------------

    mi_after = pairwise_mi_summary(
        B_test
    )

    # --------------------------------------------------------
    # Spectrum.
    # --------------------------------------------------------

    learned_sorted = np.sort(
        learned_test_entropies
    )

    true_sorted = np.sort(
        residual_entropies
    )

    low_learned = (
        learned_sorted[
            :TRUE_LOW_RANK
        ]
    )

    spectrum_error = (
        low_learned
        - true_sorted
    )

    # --------------------------------------------------------
    # Total correlation.
    # --------------------------------------------------------

    observed_TC = (
        observed_test_sum
        - learned_joint
    )

    learned_TC = (
        learned_test_sum
        - learned_joint
    )

    # ========================================================
    # RESULTS
    # ========================================================

    print()
    print("=" * 115)
    print(
        "NONLINEAR LEARNER RESULT"
    )
    print("=" * 115)

    print()

    print(
        f"selected depth = "
        f"{selected['depth']}"
    )

    print(
        f"train entropy = "
        f"{learned_train_sum:.8f}"
    )

    print(
        f"validation entropy = "
        f"{learned_validation_sum:.8f}"
    )

    print(
        f"test entropy = "
        f"{learned_test_sum:.8f}"
    )

    print()

    print(
        f"test reduction = "
        f"{100.0 * (
            1.0
            - learned_test_sum
            / observed_test_sum
        ):.6f}%"
    )

    print()

    print(
        f"population optimum = "
        f"{theoretical_optimum:.8f}"
    )

    print(
        f"finite-sample latent oracle = "
        f"{latent_test_sum:.8f}"
    )

    print(
        f"gap to finite-sample latent oracle = "
        f"{learned_test_sum - latent_test_sum:.8f}"
    )

    print(
        f"gap to population optimum = "
        f"{learned_test_sum - theoretical_optimum:.8f}"
    )

    print()

    print(
        f"exact train reversal = "
        f"{exact_train}"
    )

    print(
        f"exact test reversal = "
        f"{exact_test}"
    )

    # --------------------------------------------------------
    # Entropy spectrum.
    # --------------------------------------------------------

    print()
    print(
        "LEARNED ENTROPY SPECTRUM"
    )

    for i in np.argsort(
        learned_test_entropies
    ):

        print(
            f"  B{i:02d}: "
            f"train={learned_train_entropies[i]:.8f} "
            f"validation={learned_validation_entropies[i]:.8f} "
            f"test={learned_test_entropies[i]:.8f}"
        )

    print()
    print(
        "LOW-ENTROPY SPECTRUM"
    )

    print(
        "  planted:",
        np.array2string(
            true_sorted,
            precision=8
        )
    )

    print(
        "  learned:",
        np.array2string(
            low_learned,
            precision=8
        )
    )

    print(
        "  error:",
        np.array2string(
            spectrum_error,
            precision=8
        )
    )

    print(
        f"  spectrum MAE = "
        f"{np.mean(np.abs(spectrum_error)):.8f}"
    )

    print(
        f"  spectrum max error = "
        f"{np.max(np.abs(spectrum_error)):.8f}"
    )

    # --------------------------------------------------------
    # Pairwise MI.
    # --------------------------------------------------------

    print()
    print(
        "PAIRWISE MI AFTER"
    )

    print(
        f"  mean = "
        f"{mi_after['mean']:.8f}"
    )

    print(
        f"  median = "
        f"{mi_after['median']:.8f}"
    )

    print(
        f"  max = "
        f"{mi_after['max']:.8f}"
    )

    # --------------------------------------------------------
    # Joint entropy invariance.
    # --------------------------------------------------------

    print()
    print(
        "JOINT ENTROPY INVARIANCE"
    )

    print(
        f"  observed = "
        f"{joint_test:.8f}"
    )

    print(
        f"  learned = "
        f"{learned_joint:.8f}"
    )

    print(
        f"  difference = "
        f"{learned_joint - joint_test:.8f}"
    )

    # --------------------------------------------------------
    # Total correlation.
    # --------------------------------------------------------

    print()
    print(
        "TOTAL CORRELATION"
    )

    print(
        f"  observed = "
        f"{observed_TC:.8f}"
    )

    print(
        f"  learned = "
        f"{learned_TC:.8f}"
    )

    # ========================================================
    # FINAL COMPARISON
    # ========================================================

    print()
    print("=" * 120)
    print(
        "PAIRWISE VS LINEAR VS NONLINEAR"
    )
    print("=" * 120)

    print()

    print(
        f"{'Representation':<32}"
        f"{'Test entropy':>18}"
        f"{'Reduction':>16}"
    )

    print(
        "-" * 70
    )

    print(
        f"{'Observed':<32}"
        f"{observed_test_sum:>18.8f}"
        f"{0.0:>15.6f}%"
    )

    print(
        f"{'Best linear GF(2)':<32}"
        f"{linear['test_sum']:>18.8f}"
        f"{100.0 * (
            1.0
            - linear['test_sum']
            / observed_test_sum
        ):>15.6f}%"
    )

    print(
        f"{'Nonlinear reversible':<32}"
        f"{learned_test_sum:>18.8f}"
        f"{100.0 * (
            1.0
            - learned_test_sum
            / observed_test_sum
        ):>15.6f}%"
    )

    print(
        f"{'Latent finite-sample oracle':<32}"
        f"{latent_test_sum:>18.8f}"
        f"{100.0 * (
            1.0
            - latent_test_sum
            / observed_test_sum
        ):>15.6f}%"
    )

    print(
        f"{'Population optimum':<32}"
        f"{theoretical_optimum:>18.8f}"
        f"{100.0 * (
            1.0
            - theoretical_optimum
            / observed_test_sum
        ):>15.6f}%"
    )

    print()
    print(
        f"linear gap to population optimum = "
        f"{linear['test_sum'] - theoretical_optimum:.8f}"
    )

    print(
        f"nonlinear gap to latent oracle = "
        f"{learned_test_sum - latent_test_sum:.8f}"
    )

    # --------------------------------------------------------
    # Final success criteria.
    # --------------------------------------------------------

    # We intentionally judge nonlinear success against the
    # finite-sample latent oracle as well as population target.

    nonlinear_good = (
        learned_test_sum
        <= latent_test_sum + 0.10
    )

    exact = (
        exact_train
        and exact_test
    )

    strong_linear_gap = (
        linear["test_sum"]
        > learned_test_sum + 0.25
    )

    success = (
        nonlinear_good
        and exact
        and strong_linear_gap
    )

    print()
    print("=" * 120)

    if success:

        print(
            "NONLINEAR BENCHMARK SUCCESS"
        )

        print()
        print(
            "The nonlinear reversible learner reached "
            "the latent entropy basin while the exact "
            "linear GF(2) optimum remained substantially worse."
        )

    else:

        print(
            "PARTIAL SUCCESS"
        )

        print()
        print(
            "The learner ran successfully, but at least "
            "one strict benchmark condition was not met."
        )

    print("=" * 120)

    return {
        "Y_train": Y_train,
        "Y_validation": Y_validation,
        "Y_test": Y_test,
        "A_train": A_train,
        "A_validation": A_validation,
        "A_test": A_test,
        "B_train": B_train,
        "B_validation": B_validation,
        "B_test": B_test,
        "linear": linear,
        "learned_path": learned_path,
        "observed_test_sum":
            observed_test_sum,
        "learned_test_sum":
            learned_test_sum,
        "latent_test_sum":
            latent_test_sum,
        "theoretical_optimum":
            theoretical_optimum,
        "mi_before": mi_before,
        "mi_after": mi_after,
        "joint_test": joint_test,
        "learned_joint": learned_joint,
        "spectrum_error": spectrum_error,
        "success": success,
    }


# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    main()


CORRECTED NONLINEAR HIGHER-ORDER DEPENDENCE BENCHMARK

K = 12
TRUE LOW-ENTROPY RANK = 5
P_LOW = [0.001 0.005 0.01  0.02  0.05 ]

THEORETICAL LOW-ENTROPY CHANNELS
  Y0: p=0.001 H=0.01140776
  Y1: p=0.005 H=0.04541469
  Y2: p=0.010 H=0.08079314
  Y3: p=0.020 H=0.14144054
  Y4: p=0.050 H=0.28639696

THEORETICAL JOINT ENTROPY = 7.56545309
THEORETICAL REDUCTION = 36.954558%

PLANTED FORWARD CIRCUIT
  01. TOF(0 <- 5 & 6)
  02. TOF(1 <- 5 & 6)
  03. TOF(2 <- 5 & 6)
  04. TOF(3 <- 5 & 6)
  05. TOF(4 <- 5 & 6)
  06. CNOT(0 <- 7)
  07. CNOT(1 <- 8)
  08. CNOT(2 <- 9)
  09. CNOT(3 <- 10)
  10. CNOT(4 <- 11)

PLANTED INVERSE CIRCUIT
  01. CNOT(4 <- 11)
  02. CNOT(3 <- 10)
  03. CNOT(2 <- 9)
  04. CNOT(1 <- 8)
  05. CNOT(0 <- 7)
  06. TOF(4 <- 5 & 6)
  07. TOF(3 <- 5 & 6)
  08. TOF(2 <- 5 & 6)
  09. TOF(1 <- 5 & 6)
  10. TOF(0 <- 5 & 6)

PLANTED CIRCUIT REVERSIBILITY
train = True
test  = True

FINITE-SAMPLE LATENT ORACLE
  train entropy = 7.56683115
  test entropy = 7.56713769
  population optimum

In [5]:
# Algorithm 2
import numpy as np
import random
from math import log2

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("WARNING: PyTorch not found. Neural Baseline skipped.")

# ============================================================
# 1. REAL-WORLD DATA GENERATOR (ASCII NETWORK PROTOCOL)
# ============================================================
K = 12
N_TRAIN = 8192
N_TEST = 16384
BEAM_WIDTH = 1000  # Increased to bridge the cryptographic valley
MAX_DEPTH = 10

def generate_network_traffic_bits(num_samples):
    """
    Generates a realistic binary stream from ASCII protocol headers.
    Pairs 6-bit encoded characters into 12-bit blocks.
    """
    WORDS = ["SYSTEM", "SERVER", "KERNEL", "SECURE", "PACKET", "ROUTER", "BUFFER", "UPDATE"]
    raw_text = "".join(random.choices(WORDS, k=num_samples * 2))
    
    # Encode characters to 6-bit representations (A=0, B=1, ... Z=25)
    def char_to_6bit(c):
        if 'A' <= c <= 'Z': return ord(c) - 65
        return 26 # Space/Pad
        
    bits = []
    for c in raw_text:
        val = char_to_6bit(c)
        bits.extend([(val >> j) & 1 for j in range(6)])
        
    # Reshape into K=12 streams (2 chars per block)
    Y = np.array(bits[:num_samples * K], dtype=np.uint8).reshape(num_samples, K).T
    return Y

# ============================================================
# UTILS & INFORMATION THEORY
# ============================================================
def entropy_binary(x):
    p = float(np.mean(x))
    if p <= 0.0 or p >= 1.0: return 0.0
    return -(p * log2(p) + (1.0 - p) * log2(1.0 - p))

def apply_gate(B, gate):
    B = B.copy()
    kind = gate[0]
    if kind == "NOT": B[gate[1]] ^= 1
    elif kind == "CNOT": B[gate[1]] ^= B[gate[2]]
    elif kind == "TOF": B[gate[1]] ^= (B[gate[2]] & B[gate[3]])
    return B

POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def pack_binary(A): return np.packbits(A, axis=1, bitorder="little")
def packed_entropy(row, N):
    p = int(POP8[row].sum()) / N
    if p <= 0.0 or p >= 1.0: return 0.0
    return -(p * log2(p) + (1.0 - p) * log2(1.0 - p))

def apply_packed_gate(P, gate):
    Q = P.copy()
    kind = gate[0]
    if kind == "NOT": Q[gate[1]] ^= 0xFF
    elif kind == "CNOT": Q[gate[1]] ^= Q[gate[2]]
    elif kind == "TOF": Q[gate[1]] ^= (Q[gate[2]] & Q[gate[3]])
    return Q

# ============================================================
# ALGORITHMS
# ============================================================

def get_gate_library(K):
    gates = []
    for t in range(K): gates.append(("NOT", t))
    for t in range(K):
        for s in range(K):
            if t != s: gates.append(("CNOT", t, s))
    for t in range(K):
        for c1 in range(K):
            for c2 in range(c1 + 1, K):
                if t != c1 and t != c2: gates.append(("TOF", t, c1, c2))
    return gates

class BeamState:
    __slots__ = ("P", "entropies", "score", "path")
    def __init__(self, P, entropies, score, path):
        self.P = P; self.entropies = entropies; self.score = score; self.path = tuple(path)

def nonlinear_beam_search_lexicographical(A_train, K, beam_width, max_depth):
    """
    UPGRADED ALGORITHM: Uses Lexicographical sorting. 
    It forces the search to greedily isolate ONE pure bit at a time, 
    allowing it to bridge the cryptographic diffusion layer.
    """
    N = A_train.shape[1]
    gates = get_gate_library(K)
    P = pack_binary(A_train)
    
    init_entropies = np.array([packed_entropy(P[i], N) for i in range(K)])
    
    # Score is now a sorted tuple of entropies, not the sum!
    init_score = tuple(sorted(init_entropies))
    beam = [BeamState(P, init_entropies, init_score, ())]
    
    print(f"\n[+] Starting Lexicographical Combinatorial Search (Width={beam_width})...")
    
    for depth in range(1, max_depth + 1):
        candidates = []
        for state in beam:
            last_gate = state.path[-1] if state.path else None
            for gate in gates:
                if last_gate and gate == last_gate: continue 
                
                target = gate[1]
                cand_P = apply_packed_gate(state.P, gate)
                tgt_entropy = packed_entropy(cand_P[target], N)
                
                cand_entropies = state.entropies.copy()
                cand_entropies[target] = tgt_entropy
                
                # CRITICAL UPGRADE: Sort the entropies. Min tuple wins!
                new_score = tuple(sorted(cand_entropies))
                candidates.append(BeamState(cand_P, cand_entropies, new_score, state.path + (gate,)))
        
        # Sort candidates lexicographically (favors isolating single pure bits)
        candidates.sort(key=lambda s: s.score)
        
        # Prune beam
        unique_scores = set()
        beam = []
        for c in candidates:
            if c.score not in unique_scores: # prevent redundant paths
                unique_scores.add(c.score)
                beam.append(c)
                if len(beam) >= beam_width: break
                
        best_sum = sum(beam[0].entropies)
        print(f"  -> Depth {depth:2d}: Best Sum Entropy = {best_sum:.4f} | Purest Bit = {beam[0].score[0]:.4f}")
        
    return beam[0].path

# ============================================================
# NEURAL BASELINE WITH BIT ERROR RATE (BER)
# ============================================================
def neural_baseline(A_train, A_test, K):
    if not TORCH_AVAILABLE: return float('inf'), 1.0
    print("\n[+] Training Neural Autoencoder (Checking for Lossy Cheating)...")
    
    class DenseAE(nn.Module):
        def __init__(self, K):
            super().__init__()
            self.enc = nn.Sequential(nn.Linear(K, 64), nn.ReLU(), nn.Linear(64, K))
            self.dec = nn.Sequential(nn.Linear(K, 64), nn.ReLU(), nn.Linear(64, K))
        def forward(self, x):
            logits = self.enc(x)
            latent = torch.sigmoid(logits)
            binary_latent = (latent > 0.5).float()
            latent_st = binary_latent - latent.detach() + latent 
            return self.dec(latent_st), latent_st

    X_train = torch.tensor(A_train.T, dtype=torch.float32)
    X_test = torch.tensor(A_test.T, dtype=torch.float32)
    
    model = DenseAE(K)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    
    for epoch in range(1, 101):
        optimizer.zero_grad()
        out, latent = model(X_train)
        loss = F.binary_cross_entropy_with_logits(out, X_train) + latent.mean() * 0.1 
        loss.backward(); optimizer.step()
        if epoch % 50 == 0:
            print(f"  -> Epoch {epoch:3d} | Loss: {loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        out_test, latent_test = model(X_test)
        reconstructed = (torch.sigmoid(out_test) > 0.5).numpy().astype(np.uint8).T
        latent_test = (latent_test > 0.5).numpy().astype(np.uint8).T
    
    # Calculate Bit Error Rate (BER) - Did it corrupt the payload?
    errors = np.sum(A_test != reconstructed)
    ber = errors / (K * A_test.shape[1])
    h_test = sum([entropy_binary(latent_test[i]) for i in range(K)])
    return h_test, ber

# ============================================================
# EXECUTION
# ============================================================
def main():
    print("=" * 80)
    print("V2 CRYPTANALYSIS BENCHMARK: INTERCEPTED ASCII NETWORK PROTOCOL")
    print("=" * 80)
    
    # 1. REAL DATA
    print("\n1. Generating Real Ground Truth (ASCII Text: 'SERVER', 'KERNEL'...)")
    Y_train = generate_network_traffic_bits(N_TRAIN)
    Y_test = generate_network_traffic_bits(N_TEST)
    
    true_entropy = sum([entropy_binary(Y_test[i]) for i in range(K)])
    print(f"  -> True Text Entropy : {true_entropy:.4f} bits (Because English is predictable)")
    
    # 2. ENCRYPTION
    print("\n2. Encrypting Data via SPN (Substitution-Permutation Network)")
    cipher_gates = [
        ("TOF", 0, 5, 6), ("TOF", 1, 7, 8), ("TOF", 2, 9, 10), ("TOF", 3, 11, 4),
        ("CNOT", 5, 0), ("CNOT", 7, 1), ("CNOT", 9, 2), ("CNOT", 11, 3)
    ]
    A_train, A_test = Y_train.copy(), Y_test.copy()
    for g in cipher_gates: 
        A_train = apply_gate(A_train, g); A_test = apply_gate(A_test, g)
        
    encrypted_entropy = sum([entropy_binary(A_test[i]) for i in range(K)])
    print(f"  -> Encrypted Entropy : {encrypted_entropy:.4f} bits (Masked as White Noise)")
    
    # 3. NEURAL BASELINE
    neural_entropy, neural_ber = neural_baseline(A_train, A_test, K)
    
    # 4. BEAM SEARCH
    learned_path = nonlinear_beam_search_lexicographical(A_train, K, BEAM_WIDTH, MAX_DEPTH)
    B_test = A_test.copy()
    for gate in learned_path: B_test = apply_gate(B_test, gate)
    beam_entropy = sum([entropy_binary(B_test[i]) for i in range(K)])
    
    # ============================================================
    # RESULTS
    # ============================================================
    print("\n" + "=" * 80)
    print("FINAL BENCHMARK COMPARISON (Lower Entropy is Better, MUST have 0.0% Error)")
    print("=" * 80)
    print(f"{'Method':<32} | {'Sum of Entropies':<18} | {'Bit Error Rate (BER)'}")
    print("-" * 80)
    print(f"{'1. Encrypted Ciphertext':<32} | {encrypted_entropy:<18.4f} | 0.00% (Original)")
    
    if neural_ber > 0.01:
        print(f"{'2. Neural Baseline (PyTorch)':<32} | {neural_entropy:<18.4f} | {neural_ber*100:.2f}% (FAILED: LOSSY/CHEATED)")
    else:
        print(f"{'2. Neural Baseline (PyTorch)':<32} | {neural_entropy:<18.4f} | {neural_ber*100:.2f}% (Valid)")
        
    print(f"{'3. Non-Linear Beam Search':<32} | {beam_entropy:<18.4f} | 0.00% (STRICTLY LOSSLESS)")
    print(f"{'4. Ground Truth (English Text)':<32} | {true_entropy:<18.4f} | 0.00% (Theoretical Limit)")
    print("-" * 80)
    
    print("\n[!] GENERATED REVERSE-ENGINEERED CIRCUIT:")
    for i, gate in enumerate(learned_path): print(f"  Step {i+1:02d}: {gate}")

if __name__ == "__main__":
    main()

V2 CRYPTANALYSIS BENCHMARK: INTERCEPTED ASCII NETWORK PROTOCOL

1. Generating Real Ground Truth (ASCII Text: 'SERVER', 'KERNEL'...)
  -> True Text Entropy : 8.9992 bits (Because English is predictable)

2. Encrypting Data via SPN (Substitution-Permutation Network)
  -> Encrypted Entropy : 10.6339 bits (Masked as White Noise)

[+] Training Neural Autoencoder (Checking for Lossy Cheating)...
  -> Epoch  50 | Loss: 0.3926
  -> Epoch 100 | Loss: 0.2021

[+] Starting Lexicographical Combinatorial Search (Width=1000)...
  -> Depth  1: Best Sum Entropy = 9.7119 | Purest Bit = 0.0000
  -> Depth  2: Best Sum Entropy = 9.3035 | Purest Bit = 0.0000
  -> Depth  3: Best Sum Entropy = 9.0368 | Purest Bit = 0.0000
  -> Depth  4: Best Sum Entropy = 8.9940 | Purest Bit = 0.0000
  -> Depth  5: Best Sum Entropy = 8.8362 | Purest Bit = 0.0000
  -> Depth  6: Best Sum Entropy = 9.0464 | Purest Bit = 0.0000
  -> Depth  7: Best Sum Entropy = 8.7795 | Purest Bit = 0.0000
  -> Depth  8: Best Sum Entropy = 8.514

In [ ]:
# visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# SETTINGS FOR PUBLICATION-QUALITY PLOTS (ICML / Academic Standard)
# ==============================================================================
sns.set_theme(
    style="whitegrid", 
    context="paper", 
    font_scale=1.4, 
    rc={
        "lines.linewidth": 2.5,
        "axes.edgecolor": "black",
        "axes.linewidth": 1.2,
        "grid.linestyle": "--",
        "grid.alpha": 0.6
    }
)

# ==============================================================================
# FIGURE 1: ENTROPY SPECTRUM (Experiment 1)
# ==============================================================================
def plot_entropy_spectrum():
    # Data extracted from Experiment 1 logs
    # We take the top 5 lowest entropy coordinates to show the manifold recovery
    ranks = ["Rank 1", "Rank 2", "Rank 3", "Rank 4", "Rank 5"]
    
    planted_entropy   = [0.01140776, 0.04541469, 0.08079314, 0.14144054, 0.28639696]
    learned_entropy   = [0.00788400, 0.04404901, 0.08186752, 0.14470769, 0.28873876]
    linear_entropy    = [0.04914664, 0.09586559, 0.14017756, 0.29739187, 0.78805337]

    # Structure data for seaborn
    data = []
    for i, rank in enumerate(ranks):
        data.append({"Coordinate Rank": rank, "Entropy (Bits)": planted_entropy[i], "Representation": "Planted (Ground Truth)"})
        data.append({"Coordinate Rank": rank, "Entropy (Bits)": learned_entropy[i], "Representation": "Non-linear Reversible (Ours)"})
        data.append({"Coordinate Rank": rank, "Entropy (Bits)": linear_entropy[i], "Representation": "Exact Best Linear GF(2)"})
        
    df = pd.DataFrame(data)

    # Create figure
    plt.figure(figsize=(8, 5))
    
    # Plot grouped bar chart
    ax = sns.barplot(
        data=df, 
        x="Coordinate Rank", 
        y="Entropy (Bits)", 
        hue="Representation",
        palette=["#2ecc71", "#3498db", "#e74c3c"], # Green, Blue, Red
        edgecolor="black",
        alpha=0.9
    )

    # Aesthetics
    plt.title("Disentangled Low-Entropy Spectrum (K=12)", pad=15, fontweight="bold")
    plt.ylabel("Marginal Entropy (Bits)", fontweight="bold")
    plt.xlabel("")
    
    # Customize Legend
    plt.legend(title="", loc="upper left", frameon=True, shadow=True)
    
    # Ensure tight layout and save
    plt.tight_layout()
    plt.savefig("figure1_entropy_spectrum.pdf", format="pdf", bbox_inches="tight")
    plt.show()

# ==============================================================================
# FIGURE 2: LEXICOGRAPHICAL SEARCH TRAJECTORY (Experiment 2)
# ==============================================================================
def plot_lexicographical_trajectory():
    # Data extracted from Experiment 2 logs
    # Depth 0 is the starting encrypted cipher state
    depths = np.arange(0, 11)
    
    # Sum Entropies from the beam search at each depth step
    sum_entropies = [
        10.6652, # Depth 0 (Encrypted)
        9.7205,  # Depth 1
        9.3041,  # Depth 2
        9.0482,  # Depth 3
        8.9955,  # Depth 4
        8.7538,  # Depth 5
        9.0208,  # Depth 6 (Algorithm explores a temporary entropy increase to bridge diffusion!)
        8.2623,  # Depth 7
        7.9389,  # Depth 8
        7.6498,  # Depth 9
        7.5639   # Depth 10
    ]

    # Baseline references
    encrypted_baseline = 10.6652
    plaintext_baseline = 9.0189

    # Create figure
    plt.figure(figsize=(9, 5))

    # Plot the optimization trajectory
    sns.lineplot(
        x=depths, 
        y=sum_entropies, 
        marker="o", 
        markersize=10, 
        color="#8e44ad", 
        linewidth=3, 
        label="Lexicographical Beam Trajectory"
    )

    # Plot Baselines
    plt.axhline(
        y=encrypted_baseline, 
        color="#e74c3c", 
        linestyle="--", 
        linewidth=2.5, 
        label=f"Encrypted SPN Cipher ({encrypted_baseline:.2f} bits)"
    )
    plt.axhline(
        y=plaintext_baseline, 
        color="#27ae60", 
        linestyle="--", 
        linewidth=2.5, 
        label=f"Ground Truth English ASCII ({plaintext_baseline:.2f} bits)"
    )
    
    # Add a highlight zone where the algorithm successfully compresses BELOW standard English ASCII
    plt.fill_between(
        x=depths, 
        y1=0, 
        y2=plaintext_baseline, 
        color="#2ecc71", 
        alpha=0.1, 
        label="Super-ASCII Compression Zone"
    )

    # Aesthetics
    plt.title("Lexicographical Depth Optimization across SPN Layer", pad=15, fontweight="bold")
    plt.xlabel("Circuit Depth ($d$)", fontweight="bold")
    plt.ylabel("Sum of Entropies (Bits)", fontweight="bold")
    plt.xticks(depths)
    plt.ylim(6.5, 11.5)
    
    # Customize Legend
    plt.legend(title="", loc="lower left", frameon=True, shadow=True, fontsize=11)
    
    # Ensure tight layout and save
    plt.tight_layout()
    plt.savefig("figure2_lexicographical_trajectory.pdf", format="pdf", bbox_inches="tight")
    plt.show()

# ==============================================================================
# EXECUTE PLOTTING
# ==============================================================================
if __name__ == "__main__":
    print("Generating Figure 1: Entropy Spectrum...")
    plot_entropy_spectrum()
    
    print("Generating Figure 2: Lexicographical Optimization Trajectory...")
    plot_lexicographical_trajectory()
    
    print("Plots successfully generated and saved as PDF files.")